# Case 600 — VDI 6007 closure model and EUI comparison

This notebook makes the VDI-specific closure assumptions visible and compares annual EUI with `01_case600_pilot_validation.ipynb`. It does not tune VDI parameters to match the pilot.


In [1]:
from pathlib import Path
from dataclasses import replace
import sys
import numpy as np
import pandas as pd

cwd = Path.cwd().resolve()
preview = next((p for p in (cwd, *cwd.parents) if (p / 'case600_defaults.json').is_file()), None)
if preview is None:
    candidate = cwd / '2_validation/_BESTEST/1_notebook/preview'
    if not (candidate / 'case600_defaults.json').is_file():
        raise FileNotFoundError('Open from preview or _development.')
    preview = candidate
development = next(p for p in (preview, *preview.parents) if (p / 'RC_br/RClib/vdi6007').is_dir())
bestest = development / '2_validation/_BESTEST'
sys.path.insert(0, str(development / 'RC_br'))

from RClib.vdi6007 import RCCase, run_thermostat_case

AREA_M2 = 48.0
WEATHER = development / '2_validation/modelica_test_2/modelica-buildings-github-master/Buildings/Resources/weatherdata/USA_CO_Denver.Intl.AP.725650_TMY3.epw'
assert WEATHER.is_file(), WEATHER


## Parameter crosswalk


In [2]:
crosswalk = pd.read_csv(preview / 'case600_vdi_input_crosswalk.csv').fillna('')
display(crosswalk)
print('BESTEST-explicit rows:', (crosswalk['Explicitly mentioned in BESTEST'] == 'X').sum())
print('VDI-required rows:', (crosswalk['Required by VDI 6007'] == 'X').sum())


,Input name,Explicitly mentioned in BESTEST,Required by VDI 6007,RClib.vdi6007 implementation parameter,Numerical value / choice
0,Location / weather station,X,,RCCase.epw_path; solar weather_location,"Denver TMY3 / 39.83 deg N, -104.65 deg E"
1,Simulation year and calendar,X,,RCCase.year; hourly index,2023 surrogate non-leap year; 8760 h
2,Timestep,X,,RCCase.timestep_hours,1 h
3,Floor area,X,,RoomGeometry.floor_area_m2,48.0 m2
4,Room volume,X,,RoomGeometry.volume_m3,129.6 m3
5,Room height,X,,RoomGeometry.characteristic_height_m,2.7 m
6,North opaque wall area,X,,OpaqueComponent.area_m2,21.6 m2
7,East opaque wall area,X,,OpaqueComponent.area_m2,16.2 m2
8,South opaque wall area,X,,OpaqueComponent.area_m2,9.6 m2
9,West opaque wall area,X,,OpaqueComponent.area_m2,16.2 m2


BESTEST-explicit rows: 33
VDI-required rows: 17


## Run the annual VDI model

Inputs and outputs are controlled explicitly: fixed 8760-hour index, constant gains, fixed dual setpoints, no mechanical ventilation, and sign-separated ideal loads.


In [3]:
case = RCCase(
    year=2023,
    loc_json=preview / 'case600_location.json',
    geo_json=preview / 'case600_geometry.json',
    default_json=preview / 'case600_defaults.json',
    epw_path=WEATHER,
    occupancy_profile_csv=preview / 'case600_occupancy.csv',
    use_construction_properties=True,
)
index = pd.date_range('2023-01-01 00:00', periods=8760, freq='h')
ones = np.ones(8760)
zeros = np.zeros(8760)
simulation = run_thermostat_case(
    case,
    index=index,
    gains_w=200.0,
    occupancy_fraction=ones,
    heating_setpoint_schedule=ones * 20.0,
    cooling_setpoint_schedule=ones * 27.0,
    heating_availability_schedule=ones.astype(bool),
    cooling_availability_schedule=ones.astype(bool),
    ventilation_fraction_schedule=zeros,
    heat_recovery_efficiency_schedule=zeros,
)
net_w = np.array([hour.total_hvac_load_w for hour in simulation.hourly_results])
assert len(net_w) == 8760 and np.isfinite(net_w).all()
vdi_heating_mwh = np.clip(net_w, 0, None).sum() / 1e6
vdi_cooling_mwh = np.clip(-net_w, 0, None).sum() / 1e6
print(f'VDI heating: {vdi_heating_mwh:.6f} MWh')
print(f'VDI cooling: {vdi_cooling_mwh:.6f} MWh')
print(f'Maximum absolute balance residual: {simulation.maximum_absolute_balance_residual_w:.3e} W')


VDI heating: 3.439349 MWh
VDI cooling: 4.479599 MWh
Maximum absolute balance residual: 9.095e-13 W


## Controlled forcing: VDI + the pilot's Modelica solar CSV

This mirrors the pilot ISO experiment at the available aggregate boundary. VDI native short-wave and transmitted-window sources are disabled; the CSV's signed zone-level source is injected as radiant heat. Exterior long-wave exchange remains active. This is diagnostic, not a formal ASHRAE result.


In [4]:
modelica_solar_csv = bestest / 'results/case600/case600_iso13790_diagnostic_modelica_solar_hourly.csv'
modelica_solar = pd.read_csv(modelica_solar_csv)
assert len(modelica_solar) == 8760
assert np.array_equal(modelica_solar['timestamp_s'].to_numpy(), np.arange(1, 8761) * 3600.0)
modelica_solar_w = modelica_solar['solar_gain_W'].to_numpy(float)

def use_modelica_aggregate_solar(position, hourly):
    no_opaque_shortwave = tuple(
        replace(boundary, direct_surface_irradiance_w_m2=0.0, diffuse_surface_irradiance_w_m2=0.0)
        for boundary in hourly.boundary_inputs
    )
    return replace(
        hourly,
        boundary_inputs=no_opaque_shortwave,
        exterior_radiant_sources=(),
        internal_radiant_gain_w=hourly.internal_radiant_gain_w + float(modelica_solar_w[position]),
    )

controlled_simulation = run_thermostat_case(
    case, index=index, gains_w=200.0, occupancy_fraction=ones,
    heating_setpoint_schedule=ones * 20.0, cooling_setpoint_schedule=ones * 27.0,
    heating_availability_schedule=ones.astype(bool), cooling_availability_schedule=ones.astype(bool),
    ventilation_fraction_schedule=zeros, heat_recovery_efficiency_schedule=zeros,
    hourly_input_transform=use_modelica_aggregate_solar,
)
controlled_net_w = np.array([hour.total_hvac_load_w for hour in controlled_simulation.hourly_results])
assert len(controlled_net_w) == 8760 and np.isfinite(controlled_net_w).all()
vdi_modelica_solar_heating_mwh = np.clip(controlled_net_w, 0, None).sum() / 1e6
vdi_modelica_solar_cooling_mwh = np.clip(-controlled_net_w, 0, None).sum() / 1e6
print(f'Modelica aggregate solar integral: {modelica_solar_w.sum()/1e6:.6f} MWh')
print(f'VDI + Modelica solar heating: {vdi_modelica_solar_heating_mwh:.6f} MWh')
print(f'VDI + Modelica solar cooling: {vdi_modelica_solar_cooling_mwh:.6f} MWh')
print(f'Maximum absolute balance residual: {controlled_simulation.maximum_absolute_balance_residual_w:.3e} W')


Modelica aggregate solar integral: 12.112389 MWh
VDI + Modelica solar heating: 3.617694 MWh
VDI + Modelica solar cooling: 3.700026 MWh
Maximum absolute balance residual: 9.095e-13 W


## Compare annual energy and EUI with the pilot notebook


In [5]:
metrics = pd.read_csv(bestest / 'results/case600/case600_metrics.csv')
def pilot_value(implementation, run_mode, metric):
    row = metrics[(metrics.implementation == implementation) & (metrics.run_mode == run_mode) & (metrics.metric == metric)]
    assert len(row) == 1, (implementation, run_mode, metric)
    return float(row.value.iloc[0])

runs = [
    ('VDI 6007 closure model', vdi_heating_mwh, vdi_cooling_mwh),
    ('VDI 6007 + Modelica aggregate solar', vdi_modelica_solar_heating_mwh, vdi_modelica_solar_cooling_mwh),
    ('Pilot: Modelica native', pilot_value('modelica', 'native', 'annual_heating_energy'), pilot_value('modelica', 'native', 'annual_cooling_energy')),
    ('Pilot: ISO + Modelica solar', pilot_value('iso13790', 'diagnostic_modelica_solar', 'annual_heating_energy'), pilot_value('iso13790', 'diagnostic_modelica_solar', 'annual_cooling_energy')),
    ('Pilot: ISO native', pilot_value('iso13790', 'native', 'annual_heating_energy'), pilot_value('iso13790', 'native', 'annual_cooling_energy')),
]
comparison = pd.DataFrame(runs, columns=['Run', 'Heating (MWh)', 'Cooling (MWh)'])
comparison['Heating EUI (kWh/m2 yr)'] = comparison['Heating (MWh)'] * 1000 / AREA_M2
comparison['Cooling EUI (kWh/m2 yr)'] = comparison['Cooling (MWh)'] * 1000 / AREA_M2
base = comparison.loc[comparison.Run == 'Pilot: Modelica native'].iloc[0]
comparison['Heating vs Modelica (%)'] = (comparison['Heating (MWh)'] / base['Heating (MWh)'] - 1) * 100
comparison['Cooling vs Modelica (%)'] = (comparison['Cooling (MWh)'] / base['Cooling (MWh)'] - 1) * 100
display(comparison.round(3))


,Run,Heating (MWh),Cooling (MWh),Heating EUI (kWh/m2 yr),Cooling EUI (kWh/m2 yr),Heating vs Modelica (%),Cooling vs Modelica (%)
0,VDI 6007 closure model,3.439,4.480,71.653,93.325,-26.162,-22.801
1,VDI 6007 + Modelica aggregate solar,3.618,3.700,75.369,77.084,-22.334,-36.236
2,Pilot: Modelica native,4.658,5.803,97.042,120.889,0.000,0.000
3,Pilot: ISO + Modelica solar,4.582,5.728,95.463,119.328,-1.626,-1.291
4,Pilot: ISO native,5.502,1.414,114.633,29.454,18.127,-75.636


In [6]:
bounds = {'Heating': (3.75, 4.98), 'Cooling': (5.00, 6.83)}
status = pd.DataFrame({
    'Metric': ['Heating', 'Cooling'],
    'VDI (MWh)': [vdi_heating_mwh, vdi_cooling_mwh],
    'ASHRAE lower (MWh)': [bounds['Heating'][0], bounds['Cooling'][0]],
    'ASHRAE upper (MWh)': [bounds['Heating'][1], bounds['Cooling'][1]],
})
status['Status'] = np.where(
    status['VDI (MWh)'].between(status['ASHRAE lower (MWh)'], status['ASHRAE upper (MWh)']),
    'PASS', 'FAIL'
)
display(status.round(3))
print('Conclusion: the untuned VDI closure model is stable, but it is not yet equivalent to the pilot within the ASHRAE annual ranges.')


,Metric,VDI (MWh),ASHRAE lower (MWh),ASHRAE upper (MWh),Status
0,Heating,3.439,3.75,4.98,FAIL
1,Cooling,4.480,5.00,6.83,FAIL


Conclusion: the untuned VDI closure model is stable, but it is not yet equivalent to the pilot within the ASHRAE annual ranges.


## Interpretation

The input mapping is reproducible, but the result is not yet equivalent: both annual loads are low. Do not tune arbitrary location-based guesses to erase this gap. Prioritize sensitivity/verification of the VDI-only IW proxy, HVAC radiative split, inside film coefficient, floor boundary representation, and solar/long-wave preprocessing. The detailed parameter rationale and limitations are in `case600_vdi_validation_report.md`.
